# HCC Patient Curation List — Real-World Evidence Study

**Disease**: Hepatocellular Carcinoma (HCC) — ICD-10 C22.0  
**Data source**: Integra Connect PrecisionQ platform  
**Author**: prashanth.jain  

## Study Overview
This notebook curates a cohort of HCC patients who received 1st-line (1L) systemic therapy
within the study observation window and prepares a patient list for clinical curation.

### Cohort Criteria (Attrition Waterfall)
| Step | Criterion | N (patients) |
|------|-----------|--------------|
| 1 | C22.0 diagnosis code in disease table | ~15,190 |
| 2 | Has any 1L line of therapy on record | ~7,593 |
| 3 | 1L treatment start between **Jul 2022 – Jan 2026** | **3,010** |

### Treatment Categories (AZ Taxonomy)
- `PD-1/PD-L1_VEGF_IV` — checkpoint inhibitor + bevacizumab (e.g., atezo+beva)
- `PD-1/PD-L1_CTLA4` — dual checkpoint (e.g., durvalumab + tremelimumab)
- `PD-1/PD-L1_mono` — single checkpoint inhibitor monotherapy
- `TKI` — tyrosine kinase inhibitor (lenvatinib, sorafenib, cabozantinib, etc.)
- `chemo` — cytotoxic chemotherapy only
- `other` — rare / small-N combinations collapsed for analysis

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import datetime
import re
import sys
import types

# ── Data manipulation ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import openpyxl
from tabulate import tabulate

# ── Visualisation ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import math
import itertools
import plotly
import plotly.graph_objects as go
import plotly.offline as offline
from plotly.subplots import make_subplots
import plotly.io as pio
import imgkit

# ── Survival analysis ──────────────────────────────────────────────────────────
from lifelines import KaplanMeierFitter
from lifelines.utils import median_survival_times

# ── Statistics ─────────────────────────────────────────────────────────────────
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact

# ── Comorbidity scoring ────────────────────────────────────────────────────────
# comorbidipy uses a deprecated pandas internal; monkey-patch before importing
import pandas.errors
pandas_core_common = types.ModuleType("pandas.core.common")
pandas_core_common.SettingWithCopyWarning = pd.errors.SettingWithCopyWarning
sys.modules["pandas.core.common"] = pandas_core_common
from comorbidipy import comorbidity

# ── Presentation (disabled) ────────────────────────────────────────────────────
# from pptx import Presentation
# from pptx.util import Inches

In [ ]:
# ── Project directory structure ────────────────────────────────────────────────
# All paths live under OneDrive so the data stays synced across devices.
# Update the username below if running on a different machine.
# prashanth.jain
homedir         = r'C:\Users\prashanth.jain\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project'
datadir         = r'C:\Users\prashanth.jain\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\data'
outdir          = r'C:\Users\prashanth.jain\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\output'
interdir        = r'C:\Users\prashanth.jain\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\intermediate'
interqadir      = r'C:\Users\prashanth.jain\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\intermediateqa'
knowledgedir    = r'C:\Users\prashanth.jain\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\knowledge'
rulesdir        = r'C:\Users\prashanth.jain\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\rules'
demogr_output   = r'C:\Users\prashanth.jain\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\output\demogr'
outcomes_output = r'C:\Users\prashanth.jain\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\output\outcomes'
lot_output      = r'C:\Users\prashanth.jain\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\output\lot'

In [ ]:
def create_directories(base_dir, *sub_dirs):
    """
    Create subdirectories under base_dir if they do not already exist.

    Parameters
    ----------
    base_dir : str
        Root directory (homedir).
    *sub_dirs : str
        Full paths of subdirectories to create (not relative names).
    """
    for sub_dir in sub_dirs:
        full_path = os.path.join(base_dir, sub_dir)
        if not os.path.exists(full_path):
            os.makedirs(full_path)
            print(f"Directory created: {full_path}")
        else:
            print(f"Directory already exists: {full_path}")

In [20]:
create_directories(homedir, datadir, interdir, interqadir, knowledgedir, rulesdir, demogr_output, outcomes_output, lot_output)

Directory already exists: C:\Users\anubhav.sharma\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\data
Directory already exists: C:\Users\anubhav.sharma\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\intermediate
Directory already exists: C:\Users\anubhav.sharma\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\intermediateqa
Directory already exists: C:\Users\anubhav.sharma\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\knowledge
Directory already exists: C:\Users\anubhav.sharma\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\rules
Directory already exists: C:\Users\anubhav.sharma\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\output\demogr
Directory already exists: C:\Users\anubhav.sharma\OneDrive - Integra Connect LLC\Documents\HCC Curation List Project\output\outcomes
Directory already exists: C:\Users\anubhav.sharma\OneDrive - Integra Connect LLC\Documents\HCC Curatio

In [21]:
file_list = []
with os.scandir(datadir) as entries:
    for entry in entries:
        file_list.append(entry.name)

In [22]:
file_list

['archive',
 'archive2',
 'archive3',
 'IC_PRECISIONQ_STN_GEO_HCC_BIOMARKER_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_BIOPSY_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_COHORT_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_COMORBIDITIES_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_CONTROLTABLE_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_DEMOGRAPHICS_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_DISEASEHISTORY_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_DISEASE_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_DIVISION_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_DRUGMODIFICATIONS_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_FUTUREVISITS_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_LOTRESPONSE_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_LOT_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_MEASUREMENT_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_MEDICALCONDITION_20260114_170135.csv',
 'IC_PRECISIONQ_STN_GEO_HCC_METASTASES_2

In [23]:
csv_files = [file for file in os.listdir(datadir) if file.endswith('csv')]

In [ ]:
def change_column_names_to_lowercase(file_path):
    """
    Overwrite the header row of a CSV file with lowercase column names.

    PrecisionQ exports use UPPERCASE column names. Lowercasing them once
    at the file level avoids repeated .str.lower() calls throughout the
    analysis and keeps downstream code consistent.

    Parameters
    ----------
    file_path : str
        Absolute path to the CSV file to modify in-place.
    """
    with open(file_path, 'r', newline='') as csvfile:
        header = csvfile.readline()

    lowercase_header = header.lower()

    with open(file_path, 'r+', newline='') as csvfile:
        csvfile.seek(0)
        csvfile.write(lowercase_header)


# Apply to every CSV in datadir
for csv_file in csv_files:
    file_path = os.path.join(datadir, csv_file)

change_column_names_to_lowercase(file_path)

In [ ]:
# ── Resolve file names from directory listing ──────────────────────────────────
# The extract date stamp in the filename changes with each pull, so we match
# by keyword rather than hardcoding the full filename.
demogr_file      = [f for f in file_list if 'DEMOGRAPHICS'  in f][0]  # patient-level demographics
lot_file         = [f for f in file_list if 'LOT_'          in f][0]  # lines of therapy
disease_file     = [f for f in file_list if 'DISEASE_'      in f][0]  # primary disease record (one row per patient)
diseasehistory_file = [f for f in file_list if 'DISEASEHISTORY' in f][0]  # longitudinal diagnosis history
biomarker_file   = [f for f in file_list if 'BIOMARKER'     in f][0]  # biomarker results
visit_file       = [f for f in file_list if 'VISIT'         in f][0]  # clinic visit records (FUTUREVISITS)
procedure_file   = [f for f in file_list if 'PROCEDURE'     in f][0]  # procedures (imaging, surgery, etc.)
comorb_file      = [f for f in file_list if 'COMORB'        in f][0]  # comorbidity conditions

## 1. Load Data

Load all PrecisionQ CSVs into dataframes. Date columns are parsed at load time
so all downstream comparisons use `datetime64` consistently.

In [ ]:
# Demographics — one row per patient; contains age, gender, race, payer, curation status
df_demogr = pd.read_csv(os.path.join(datadir, demogr_file), header=0)

In [ ]:
# Lines of Therapy — one row per patient per LOT; contains regimen, start/end dates, metastatic flag
df_lot = pd.read_csv(os.path.join(datadir, lot_file), header=0)

In [ ]:
# Parse LOT date columns from string to datetime
df_lot['start_date'] = df_lot['start_date'].apply(pd.to_datetime)
df_lot['end_date']   = df_lot['end_date'].apply(pd.to_datetime)

In [29]:
df_lot.head()

,mpi_id,division_mask,combined_div_mpi_id,lot,regimen,start_date,end_date,duration_days,duration_months,metastatic,no_div_lot
0,100316640,128,128_100316640,1,sorafenib,2018-11-19,2019-04-18,150,5,0,1
1,100316640,128,128_100316640,2,nivolumab,2019-04-19,2019-04-19,0,0,0,2
2,106257467,128,128_106257467,1,"cisplatin,gemcitabine hydrochloride",2013-09-26,2014-01-15,111,4,0,1
3,106257467,128,128_106257467,2,"gemcitabine hydrochloride,oxaliplatin",2014-01-30,2014-01-30,0,0,0,2
4,107245745,128,128_107245745,1,"lenvatinib,sorafenib",2020-08-28,2020-10-02,35,2,0,1


In [ ]:
# Disease — primary HCC record per patient; contains cancer_code, stage, TNM, histology, metastatic info
df_disease = pd.read_csv(os.path.join(datadir, disease_file), header=0)

In [ ]:
df_disease['diag_date'] = df_disease['diag_date'].apply(pd.to_datetime)

In [ ]:
# Disease History — longitudinal diagnosis table; multiple rows per patient across time.
# Used alongside df_disease to catch patients whose primary code may differ.
df_diseasehistory = pd.read_csv(os.path.join(datadir, diseasehistory_file), header=0)
df_diseasehistory['diag_date'] = pd.to_datetime(df_diseasehistory['diag_date'], format='%Y-%m-%d')

In [ ]:
# Comorbidities — ICD-coded concurrent conditions with start/end dates
df_comorb = pd.read_csv(os.path.join(datadir, comorb_file), header=0)
df_comorb['cond_st_date']  = pd.DatetimeIndex(df_comorb['cond_st_date'])
df_comorb['cond_end_date'] = pd.DatetimeIndex(df_comorb['cond_end_date'])

In [ ]:
# Visits — all clinic encounters; used to verify activity/follow-up within study window
df_visit = pd.read_csv(os.path.join(datadir, visit_file), header=0)
df_visit['visit_date'] = pd.DatetimeIndex(df_visit['visit_date'])
df_visit.head()

In [ ]:
# Procedures — imaging (CT, MRI), surgeries, transplants, TACE, biopsies, etc.
# low_memory=False suppresses mixed-type warning on column 25
df_procedure = pd.read_csv(os.path.join(datadir, procedure_file), header=0, low_memory=False)
df_procedure['date_event'] = pd.DatetimeIndex(df_procedure['date_event'])

## 2. Define Cohort

Apply inclusion/exclusion criteria to arrive at the analysis cohort.
Steps mirror the attrition waterfall documented in the project overview above.

In [36]:
df_procedure.head(1)

,mpi_id,division_mask,combined_div_mpi_id,date_event,procedure,procedure_source_code,value,location,units,transplant_type,...,resection_status,rsi,procedure_concept_id,date_end,stemcell_collection_method,stemcell_collection_date,location_suv,current_suv,prior_suv,laterality
0,112171181,128,128_112171181,2024-12-10,CT Scan,NaN,Local Disease Only,NaN,NaN,NaN,...,NaN,Z,0,2024-12-10,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# EXCLUSION CHECK — Clinical trial participation
# Procedure source code 'S99' indicates a clinical trial visit.
# Any patient with an S99 code on or after the study start (2022-11-01) is excluded.
# Result is expected to be empty (no such patients in current extract).
df_procedure[
    df_procedure['procedure_source_code'].str.contains('S99', na=False) &
    (df_procedure['date_event'] >= '2022-11-01')
]

In [38]:
df_diseasehistory[df_diseasehistory['cancer_code'].str.contains('Z00')]

,mpi_id,division_mask,combined_div_mpi_id,diag_date,cancer_code,in_remission,in_relapse,date_of_progression,rsi,condition_concept_id


### 2a. Identify HCC patients — ICD-10 C22.0

C22.0 = Hepatocellular carcinoma of liver.  
We pull from **two** sources and take the union:
- `df_disease` — the primary curated disease record (most reliable, one row per patient)  
- `df_diseasehistory` — longitudinal history (catches patients coded only in history, e.g. coded as C22.8 in disease but C22.0 appears in history)

In [39]:
df_lot['mpi_id'].nunique()

10571

In [ ]:
# Unique mpi_ids with C22.0 anywhere in their diagnosis history
diag_history_code_pt = (
    df_diseasehistory[df_diseasehistory['cancer_code'] == 'C22.0']['mpi_id']
    .drop_duplicates()
)

In [41]:
diag_history_code_pt.nunique()

15566

In [ ]:
# Unique mpi_ids with C22.0 as their primary disease code
disease_diag_code_pt = (
    df_disease[df_disease['cancer_code'] == 'C22.0']['mpi_id']
    .drop_duplicates()
)

In [43]:
disease_diag_code_pt.nunique()

15190

In [44]:
disease_diag_code_pt[disease_diag_code_pt.isin(diag_history_code_pt)]

0        355000716
1        356830665
2        357503542
4        358644266
5        360849566
           ...    
21701    500387928
21703    508166433
21704    510665375
21705    511952086
21706    513325027
Name: mpi_id, Length: 15190, dtype: int64

In [45]:
df_diseasehistory[df_diseasehistory['mpi_id'].isin(diag_history_code_pt[~diag_history_code_pt.isin(disease_diag_code_pt)])]

,mpi_id,division_mask,combined_div_mpi_id,diag_date,cancer_code,in_remission,in_relapse,date_of_progression,rsi,condition_concept_id
90,149866628,997,997_149866628,2014-01-01,C22.8,NaN,NaN,NaN,EP,45585987
91,149866628,997,997_149866628,2014-01-22,C22.0,NaN,NaN,NaN,P,45585987
92,149866628,997,997_149866628,2016-04-21,C22.8,NaN,NaN,NaN,P,45585987
1957,188660784,997,997_188660784,2020-02-03,C22.1,NaN,NaN,NaN,P,45585987
1958,188660784,997,997_188660784,2020-02-07,C22.8,NaN,NaN,NaN,EP,45585987
...,...,...,...,...,...,...,...,...,...,...
326961,571742195,316,316_571742195,2019-05-03,C22.8,NaN,NaN,NaN,P,45585987
326962,571742195,316,316_571742195,2019-05-31,C22.8,NaN,NaN,NaN,P,45585987
326963,571742195,316,316_571742195,2019-08-16,C22.8,NaN,NaN,NaN,P,45585987
326964,571742195,316,316_571742195,2019-08-27,C22.0,NaN,NaN,NaN,P,45585987


In [46]:
df_diseasehistory[df_diseasehistory['mpi_id'].isin(diag_history_code_pt[~diag_history_code_pt.isin(disease_diag_code_pt)])]

,mpi_id,division_mask,combined_div_mpi_id,diag_date,cancer_code,in_remission,in_relapse,date_of_progression,rsi,condition_concept_id
90,149866628,997,997_149866628,2014-01-01,C22.8,NaN,NaN,NaN,EP,45585987
91,149866628,997,997_149866628,2014-01-22,C22.0,NaN,NaN,NaN,P,45585987
92,149866628,997,997_149866628,2016-04-21,C22.8,NaN,NaN,NaN,P,45585987
1957,188660784,997,997_188660784,2020-02-03,C22.1,NaN,NaN,NaN,P,45585987
1958,188660784,997,997_188660784,2020-02-07,C22.8,NaN,NaN,NaN,EP,45585987
...,...,...,...,...,...,...,...,...,...,...
326961,571742195,316,316_571742195,2019-05-03,C22.8,NaN,NaN,NaN,P,45585987
326962,571742195,316,316_571742195,2019-05-31,C22.8,NaN,NaN,NaN,P,45585987
326963,571742195,316,316_571742195,2019-08-16,C22.8,NaN,NaN,NaN,P,45585987
326964,571742195,316,316_571742195,2019-08-27,C22.0,NaN,NaN,NaN,P,45585987


In [ ]:
# Patients who appear in disease history with C22.0 but NOT in the primary disease table.
# Take the earliest C22.0 history record as their index diagnosis date.
df_diseasehistory_c220 = (
    df_diseasehistory[
        df_diseasehistory['mpi_id'].isin(diag_history_code_pt[~diag_history_code_pt.isin(disease_diag_code_pt)]) &
        (df_diseasehistory['cancer_code'] == 'C22.0')
    ]
    .sort_values(by=['mpi_id', 'diag_date'], ascending=True)
    .drop_duplicates('mpi_id', keep='first')
)

In [48]:
df_diseasehistory['cancer_code'].value_counts().head(20)

cancer_code
C22.0     137689
C22.8      24065
C79.51     11818
C22.1      10354
155.0       9951
C78.7       7831
C61         5862
C22.9       4617
C90.00      3379
C18.9       3049
C34.11      2925
C80.1       2853
D47.2       2623
C20         2352
D46.9       2331
R97.8       2293
C34.90      2208
C78.00      2165
C25.9       2094
R18.8       1602
Name: count, dtype: int64

In [49]:
df_disease[df_disease['mpi_id'].isin(diag_history_code_pt[~diag_history_code_pt.isin(disease_diag_code_pt)])]

,mpi_id,division_mask,combined_div_mpi_id,diag_date,cancer_code,cancer_stage,t_value,n_value,m_value,g_value,...,date_of_progression,metastatic_date,metastatic_site,disease_subtype,location,histology,crpc_diagnosis_date,rsi,condition_concept_id,mcrpc_date
34,385323119,316,316_385323119,2011-01-01,155.0,I,T1,N0,M0,NaN,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,S,44819423,NaN
98,42898523,316,316_42898523,2017-08-30,C22.8,NaN,T2,N0,M0,GX,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,E,45585987,NaN
150,463933210,316,316_463933210,2017-06-14,C22.8,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Liver,NaN,NaN,NaN,45585987,NaN
291,571742195,316,316_571742195,2018-10-02,C22.8,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Liver,NaN,NaN,NaN,45585987,NaN
438,673561559,316,316_673561559,2015-06-22,C22.8,I,T1,N0,M0,NaN,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,E,45585987,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21563,253022690,773,773_253022690,2019-04-09,C22.8,II,T2,N0,M0,NaN,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,D,45585987,NaN
21618,366843469,773,773_366843469,2019-06-06,C22.8,IVB,T4,NaN,M1,NaN,...,NaN,2019-06-06,NaN,NaN,Liver,Unknown,NaN,D,45585987,NaN
21630,388970372,773,773_388970372,2018-10-15,C22.8,IVB,T,N,M1,NaN,...,NaN,2020-08-04,NaN,NaN,Liver,Unknown,NaN,D,45585987,NaN
21678,464588306,773,773_464588306,2017-12-05,C22.8,II,T2,N0,M0,NaN,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,D,45585987,NaN


In [ ]:
# Primary C22.0 cohort — patients with C22.0 in the main disease table.
# This is the base for all downstream cohort filtering.
df_disease_diag_use = df_disease[df_disease['cancer_code'] == 'C22.0']

In [51]:
df_disease_diag_use['cancer_code'].value_counts()

cancer_code
C22.0    15489
Name: count, dtype: int64

In [52]:
### attrition 1: total number of patients with C22.0 and diagnosis code rsi have A or E
df_demogr_use = df_demogr[(df_demogr['mpi_id'].isin(df_lot['mpi_id']))]
df_demogr_use

,mpi_id,flag,division_mask,combined_div_mpi_id,gender,age_dx,age_tx,payer,race,ethnicity,...,last_cur_abandon_date,cur_type,rsi,gender_concept_id,race_concept_id,ethnicity_concept_id,specialty_concept_id,menopausal_status,last_activity,other_primary
0,802583372,1,336,336_802583372,MALE,78,78.0,Medicare/Medicaid,White,Not Hispanic or Latino,...,NaN,NaN,E,8507,8527,38003564,0,NaN,2024-11-01,0
1,815029852,1,336,336_815029852,MALE,71,71.0,Other,White,Not Hispanic or Latino,...,NaN,NaN,E,8507,8527,38003564,0,NaN,2023-04-27,1
5,857828353,1,336,336_857828353,FEMALE,59,60.0,Other,Other,Hispanic or Latino,...,NaN,NaN,E,8532,0,38003563,0,NaN,2019-05-24,0
7,888943278,1,336,336_888943278,MALE,75,76.0,Other,White,Not Hispanic or Latino,...,NaN,NaN,E,8507,8527,38003564,0,NaN,2025-12-23,1
8,892176674,1,336,336_892176674,MALE,57,57.0,Other,White,Not Hispanic or Latino,...,NaN,NaN,E,8507,8527,38003564,0,NaN,2020-08-24,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21698,647996562,1,713,713_647996562,MALE,64,64.0,Medicare,Other,Other,...,NaN,NaN,EP,8507,0,0,0,NaN,2025-04-25,1
21699,648604275,1,713,713_648604275,MALE,71,71.0,Medicare,Other,Other,...,NaN,NaN,EP,8507,0,0,0,NaN,2009-11-06,0
21700,648859621,1,713,713_648859621,MALE,78,78.0,Commercial,Other,Not Hispanic or Latino,...,NaN,NaN,EP,8507,0,38003564,0,NaN,2020-08-25,1
21704,64966038,1,713,713_64966038,FEMALE,74,74.0,Other,Other,Not Hispanic or Latino,...,NaN,NaN,E,8532,0,38003564,0,NaN,2013-07-18,1


In [53]:
df_demogr_use['mpi_id'].nunique()

10571

In [54]:
# Step 1: Identify all mpi_ids matching the condition
excluded_ids = df_demogr_use[
    df_demogr_use['last_cur_abandon_date'].notnull() &
    df_demogr_use['last_cur_completion_date'].isnull()
]['mpi_id'].unique()

# Step 2: Exclude all rows with those mpi_ids
df_demogr_use = df_demogr_use[~df_demogr_use['mpi_id'].isin(excluded_ids)]

df_demogr_use

,mpi_id,flag,division_mask,combined_div_mpi_id,gender,age_dx,age_tx,payer,race,ethnicity,...,last_cur_abandon_date,cur_type,rsi,gender_concept_id,race_concept_id,ethnicity_concept_id,specialty_concept_id,menopausal_status,last_activity,other_primary
0,802583372,1,336,336_802583372,MALE,78,78.0,Medicare/Medicaid,White,Not Hispanic or Latino,...,NaN,NaN,E,8507,8527,38003564,0,NaN,2024-11-01,0
1,815029852,1,336,336_815029852,MALE,71,71.0,Other,White,Not Hispanic or Latino,...,NaN,NaN,E,8507,8527,38003564,0,NaN,2023-04-27,1
5,857828353,1,336,336_857828353,FEMALE,59,60.0,Other,Other,Hispanic or Latino,...,NaN,NaN,E,8532,0,38003563,0,NaN,2019-05-24,0
7,888943278,1,336,336_888943278,MALE,75,76.0,Other,White,Not Hispanic or Latino,...,NaN,NaN,E,8507,8527,38003564,0,NaN,2025-12-23,1
8,892176674,1,336,336_892176674,MALE,57,57.0,Other,White,Not Hispanic or Latino,...,NaN,NaN,E,8507,8527,38003564,0,NaN,2020-08-24,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21698,647996562,1,713,713_647996562,MALE,64,64.0,Medicare,Other,Other,...,NaN,NaN,EP,8507,0,0,0,NaN,2025-04-25,1
21699,648604275,1,713,713_648604275,MALE,71,71.0,Medicare,Other,Other,...,NaN,NaN,EP,8507,0,0,0,NaN,2009-11-06,0
21700,648859621,1,713,713_648859621,MALE,78,78.0,Commercial,Other,Not Hispanic or Latino,...,NaN,NaN,EP,8507,0,38003564,0,NaN,2020-08-25,1
21704,64966038,1,713,713_64966038,FEMALE,74,74.0,Other,Other,Not Hispanic or Latino,...,NaN,NaN,E,8532,0,38003564,0,NaN,2013-07-18,1


In [55]:
df_demogr_use['mpi_id'].nunique()

10571

In [56]:
df_demogr_use['last_cur_abandon_date'].isnull().sum()

10742

In [ ]:
# ATTRITION STEP 1 — Patients with confirmed C22.0 diagnosis (n ≈ 15,190)
# Restrict demographics table to patients who have a C22.0 entry in df_disease.
df_demogr_use = df_demogr[df_demogr['mpi_id'].isin(df_disease_diag_use['mpi_id'])]

In [58]:
### no diag code restriction
df_demogr[df_demogr['mpi_id'].isin(df_disease['mpi_id'])]['mpi_id'].nunique()

21275

In [59]:
df_disease['mpi_id'].nunique()

21275

In [60]:
df_disease[~df_disease['mpi_id'].isin(df_disease_diag_use['mpi_id'])]['cancer_code'].value_counts()

cancer_code
155.0    3302
C22.8    2876
Name: count, dtype: int64

In [61]:
df_demogr_use['mpi_id'].nunique()

15190

In [ ]:
# ATTRITION STEP 2 — Patients with at least one line of therapy (n ≈ 7,593)
# no_div_lot == 1 means this row is the patient's 1st line within their division.
df_lot_demogr_use = df_lot[
    (df_lot['no_div_lot'] == 1) &
    (df_lot['mpi_id'].isin(df_demogr_use['mpi_id']))
]
print(df_lot_demogr_use['mpi_id'].nunique())

In [ ]:
# ATTRITION STEP 3 — 1L treatment start within study window: Jul 2022 – Jan 2026 (n = 3,010)
# Study window aligns with durvalumab+tremelimumab approval (Oct 2022) plus a 3-month lead-in.
df_lot[
    (df_lot['no_div_lot'] == 1) &
    (df_lot['start_date'] >= '2022-07-01') &
    (df_lot['start_date'] <= '2026-01-31') &
    (df_lot['mpi_id'].isin(df_demogr_use['mpi_id']))
]['mpi_id'].nunique()

In [ ]:
# Build the final LOT dataset for the 3,010-patient cohort.
# Step 1: identify qualifying patients via their 1L row.
df_lot1 = df_lot[
    (df_lot['no_div_lot'] == 1) &
    (df_lot['start_date'] >= '2022-07-01') &
    (df_lot['start_date'] <= '2026-01-31') &
    (df_lot['mpi_id'].isin(df_demogr_use['mpi_id']))
]

# Step 2: pull ALL LOT rows for those patients (LOT1 through LOT-n) so
# subsequent-line treatment sequences can be analysed later.
df_lot_use = df_lot[df_lot['mpi_id'].isin(df_lot1['mpi_id'])]

In [65]:
df_demogr_use['age_dx'].min()

19

In [66]:
df_lot_use['mpi_id'].nunique()

3010

In [67]:
df_lot_use['no_div_lot'].value_counts()

no_div_lot
1    3010
2     723
3     184
4      48
5      12
6       3
7       1
Name: count, dtype: int64

In [68]:
df_lot_use['start_date'].min()

Timestamp('2022-07-01 00:00:00')

In [69]:
df_lot_use['regimen'].value_counts()

regimen
durvalumab,tremelimumab-actl                 538
atezolizumab,bevacizumab-bvzr                364
durvalumab,tremelimumab,tremelimumab-actl    354
atezolizumab,bevacizumab-awwb                334
lenvatinib                                   318
                                            ... 
atezolizumab,bevacizumab,fluorouracil          1
cabozantinib,lenvatinib,sorafenib,tace         1
capecitabine,lenvatinib,nivolumab              1
doxorubicin,lenvatinib,tace                    1
durvalumab,tae                                 1
Name: count, Length: 308, dtype: int64

In [70]:
df_lot_use[df_lot_use['regimen'].str.contains('bevacizumab')][['regimen']].value_counts().head(10)

regimen                                                   
atezolizumab,bevacizumab-bvzr                                 364
atezolizumab,bevacizumab-awwb                                 334
atezolizumab,bevacizumab-maly                                 113
atezolizumab,bevacizumab-adcd                                 100
atezolizumab,bevacizumab                                       68
atezolizumab,bevacizumab-awwb,bevacizumab-bvzr                 58
atezolizumab,bevacizumab,bevacizumab-awwb,bevacizumab-bvzr     49
atezolizumab,bevacizumab,bevacizumab-awwb                      42
atezolizumab,bevacizumab,bevacizumab-bvzr                      24
atezolizumab,bevacizumab-bvzr,bevacizumab-maly                 23
Name: count, dtype: int64

In [71]:
df_lot_use[df_lot_use['regimen'].str.contains('-')]['regimen'].drop_duplicates()

6                             durvalumab,tremelimumab-actl
25                           atezolizumab,bevacizumab-bvzr
38                           atezolizumab,bevacizumab-maly
65          atezolizumab,bevacizumab-bvzr,bevacizumab-maly
66              atezolizumab,bevacizumab-bvzr,cabozantinib
                               ...                        
13902    atezolizumab,bevacizumab,bevacizumab-awwb,beva...
13999    atezolizumab,bevacizumab,bevacizumab-awwb,beva...
14114    durvalumab,lenvatinib,nivolumab,tremelimumab-actl
14434    cabozantinib,cabozantinib s-malate,ipilimumab,...
14472    atezolizumab,bevacizumab,bevacizumab-awwb,beva...
Name: regimen, Length: 143, dtype: object

In [ ]:
def replace_biosimilars(regimen):
    """
    Standardise a regimen string by stripping biosimilar suffixes.

    PrecisionQ records biosimilars with a dash suffix (e.g., bevacizumab-bvzr,
    bevacizumab-awwb, tremelimumab-actl). This function collapses all variants
    to the reference molecule name so counts are not fragmented across suffixes.

    Special case: any trastuzumab variant returns 'trastuzumab' alone
    (not relevant for HCC but retained for safety).

    Parameters
    ----------
    regimen : str
        Comma-separated drug string from the LOT table.

    Returns
    -------
    str
        Comma-separated, alphabetically sorted, deduplicated reference drug names.

    Examples
    --------
    'atezolizumab,bevacizumab-bvzr,bevacizumab-awwb' -> 'atezolizumab,bevacizumab'
    'durvalumab,tremelimumab-actl'                    -> 'durvalumab,tremelimumab'
    """
    drugs = set(drug.strip() for drug in regimen.split(','))
    standardized_drugs = {drug.split('-')[0] for drug in drugs}
    if 'trastuzumab' in standardized_drugs or any('trastuzumab' in drug for drug in drugs):
        return 'trastuzumab'
    return ','.join(sorted(standardized_drugs))

In [73]:
df_lot_use['regimen_biosimilar'] = df_lot_use['regimen'].apply(replace_biosimilars)

In [74]:
df_lot_use

,mpi_id,division_mask,combined_div_mpi_id,lot,regimen,start_date,end_date,duration_days,duration_months,metastatic,no_div_lot,regimen_biosimilar
6,112171181,128,128_112171181,1,"durvalumab,tremelimumab-actl",2025-08-07,2025-10-30,84,2,1,1,"durvalumab,tremelimumab"
15,137244673,128,128_137244673,1,"durvalumab,tremelimumab-actl",2023-12-07,2025-12-05,729,24,1,1,"durvalumab,tremelimumab"
25,244067723,128,128_244067723,1,"atezolizumab,bevacizumab-bvzr",2023-03-08,2023-04-26,49,1,1,1,"atezolizumab,bevacizumab"
26,252301890,128,128_252301890,1,"atezolizumab,bevacizumab-bvzr",2022-11-02,2023-06-21,231,7,1,1,"atezolizumab,bevacizumab"
27,255272953,128,128_255272953,1,sorafenib,2023-07-11,2023-09-09,60,2,0,1,sorafenib
...,...,...,...,...,...,...,...,...,...,...,...,...
14553,995045745,997,997_995045745,1,atezolizumab,2023-04-03,2023-07-10,98,3,0,1,atezolizumab
14554,995045745,997,997_995045745,2,"durvalumab,tremelimumab-actl",2023-08-02,2024-01-30,181,5,0,2,"durvalumab,tremelimumab"
14555,995045745,997,997_995045745,3,lenvatinib,2024-02-20,2024-10-05,228,8,0,3,lenvatinib
14561,997668559,997,997_997668559,1,nivolumab,2022-07-11,2022-10-04,85,3,0,1,nivolumab


In [ ]:
def categorize_regimen(regimen):
    """
    Map a standardised regimen string to an AZ treatment category.

    Drug class membership is determined by set intersection. The hierarchy
    is PD-1/PD-L1 > TKI > VEGF_IV > CTLA4 > chemo > other, so a patient on
    atezo + beva + lenvatinib would be classified as PD-1/PD-L1_VEGF_IV_TKI
    rather than just TKI.

    Parameters
    ----------
    regimen : str
        Comma-separated, biosimilar-standardised drug string
        (output of replace_biosimilars).

    Returns
    -------
    str
        One of the following category labels:
        - PD-1/PD-L1_mono          : checkpoint inhibitor alone
        - PD-1/PD-L1_CTLA4         : dual checkpoint (e.g., durva + tremi)
        - PD-1/PD-L1_VEGF_IV       : checkpoint + anti-VEGF IV (e.g., atezo + beva)
        - PD-1/PD-L1_TKI           : checkpoint + TKI
        - PD-1/PD-L1_chemo         : checkpoint + chemotherapy
        - PD-1/PD-L1_VEGF_IV_TKI   : checkpoint + beva + TKI
        - PD-1/PD-L1_CTLA4_VEGF_IV : dual checkpoint + beva
        - PD-1/PD-L1_CTLA4_TKI     : dual checkpoint + TKI
        - PD-1/PD-L1_VEGF_IV_chemo : checkpoint + beva + chemo
        - PD-1/PD-L1_TKI_chemo     : checkpoint + TKI + chemo
        - PD-1/PD-L1_other         : checkpoint + unclassified combination
        - TKI                       : TKI monotherapy or TKI + non-IO
        - TKI_chemo                 : TKI + chemotherapy
        - VEGF_IV                   : bevacizumab/ramucirumab alone
        - VEGF_IV_chemo             : VEGF_IV + chemotherapy
        - CTLA4                     : CTLA-4 inhibitor without PD-1/PD-L1
        - chemo                     : cytotoxic only
        - other                     : does not fit any category above
    """
    # Drug class definitions
    pd1_pd_l1 = {'nivolumab', 'pembrolizumab', 'atezolizumab', 'durvalumab',
                 'dostarlimab', 'dostarlimab-gxly', 'toripalimab'}
    ctla4     = {'ipilimumab', 'tremelimumab', 'tremelimumab-actl'}
    tki       = {'lenvatinib', 'sorafenib', 'cabozantinib', 'regorafenib',
                 'selpercatinib', 'larotrectinib', 'entrectinib'}
    vegf_iv   = {'ramucirumab', 'bevacizumab'}
    chemo     = {'fluorouracil', 'gemcitabine', 'capecitabine', 'oxaliplatin', 'cisplatin'}

    drugs = set(drug.strip() for drug in regimen.split(','))

    has_pd1_pd_l1 = bool(drugs & pd1_pd_l1)
    has_ctla4     = bool(drugs & ctla4)
    has_tki       = bool(drugs & tki)
    has_vegf_iv   = bool(drugs & vegf_iv)
    has_chemo     = bool(drugs & chemo)

    if has_pd1_pd_l1:
        if drugs <= pd1_pd_l1:                         return 'PD-1/PD-L1_mono'
        elif has_ctla4 and has_vegf_iv and has_tki:    return 'PD-1/PD-L1_CTLA4_VEGF_IV_TKI'
        elif has_ctla4 and has_vegf_iv:                return 'PD-1/PD-L1_CTLA4_VEGF_IV'
        elif has_ctla4 and has_tki:                    return 'PD-1/PD-L1_CTLA4_TKI'
        elif has_ctla4:                                return 'PD-1/PD-L1_CTLA4'
        elif has_vegf_iv and has_tki:                  return 'PD-1/PD-L1_VEGF_IV_TKI'
        elif has_vegf_iv:
            return 'PD-1/PD-L1_VEGF_IV_chemo' if has_chemo else 'PD-1/PD-L1_VEGF_IV'
        elif has_tki:
            return 'PD-1/PD-L1_TKI_chemo' if has_chemo else 'PD-1/PD-L1_TKI'
        elif has_chemo:                                return 'PD-1/PD-L1_chemo'
        else:                                          return 'PD-1/PD-L1_other'
    elif has_tki:
        return 'TKI_chemo' if has_chemo else 'TKI'
    elif has_vegf_iv:
        return 'VEGF_IV_chemo' if has_chemo else 'VEGF_IV'
    elif has_ctla4:    return 'CTLA4'
    elif has_chemo:    return 'chemo'
    else:              return 'other'

In [76]:
df_lot_use_cat = df_lot_use
df_lot_use_cat['AZ_category_use'] = df_lot_use_cat['regimen_biosimilar'].apply(categorize_regimen)

In [77]:
df_lot_use_cat['no_div_lot'].value_counts()

no_div_lot
1    3010
2     723
3     184
4      48
5      12
6       3
7       1
Name: count, dtype: int64

In [78]:
df_regimen_cat_pt = df_lot_use_cat[df_lot_use_cat['no_div_lot']==1][['mpi_id', 'regimen_biosimilar','AZ_category_use']]

In [ ]:
# Collapse low-frequency / analytically unusable categories into 'other'.
# Categories below are either too small for robust statistics, clinically
# heterogeneous, or off-label combinations not relevant to the AZ analysis.
RARE_CATEGORIES = [
    'CTLA4',                  # CTLA-4 mono without PD-1 — very small N
    'PD-1/PD-L1_VEGF_IV_chemo',
    'PD-1/PD-L1_CTLA4_TKI',
    'TKI_chemo',
    'VEGF_IV',                # IV VEGF without IO
    'PD-1/PD-L1_TKI_chemo',
    'other',
]

df_regimen_cat_pt['AZ_category_use'] = df_regimen_cat_pt['AZ_category_use']
df_regimen_cat_pt.loc[
    df_regimen_cat_pt['AZ_category_use'].isin(RARE_CATEGORIES), 'AZ_category_use'
] = 'other'

In [80]:
df_lot_use_cat['regimen_biosimilar'].value_counts()

regimen_biosimilar
atezolizumab,bevacizumab                                        1237
durvalumab,tremelimumab                                          937
lenvatinib                                                       318
cabozantinib                                                     186
durvalumab                                                       125
                                                                ... 
bevacizumab,durvalumab,fluorouracil,irinotecan hydrochloride       1
bevacizumab,durvalumab,fluorouracil,oxaliplatin                    1
bevacizumab,cisplatin,durvalumab,gemcitabine                       1
cabozantinib,pembrolizumab                                         1
durvalumab,tae                                                     1
Name: count, Length: 231, dtype: int64

In [81]:
df_regimen_cat_pt

,mpi_id,regimen_biosimilar,AZ_category_use
6,112171181,"durvalumab,tremelimumab",PD-1/PD-L1_CTLA4
15,137244673,"durvalumab,tremelimumab",PD-1/PD-L1_CTLA4
25,244067723,"atezolizumab,bevacizumab",PD-1/PD-L1_VEGF_IV
26,252301890,"atezolizumab,bevacizumab",PD-1/PD-L1_VEGF_IV
27,255272953,sorafenib,TKI
...,...,...,...
14547,989569413,lenvatinib,TKI
14552,994201021,"cisplatin,durvalumab,gemcitabine,gemcitabine h...",PD-1/PD-L1_chemo
14553,995045745,atezolizumab,PD-1/PD-L1_mono
14561,997668559,nivolumab,PD-1/PD-L1_mono


In [82]:
df_regimen_cat_pt['AZ_category_use'].value_counts()

AZ_category_use
PD-1/PD-L1_VEGF_IV          1191
PD-1/PD-L1_CTLA4             920
PD-1/PD-L1_mono              285
TKI                          229
other                        135
chemo                        115
PD-1/PD-L1_chemo              80
PD-1/PD-L1_other              14
PD-1/PD-L1_VEGF_IV_TKI        13
PD-1/PD-L1_TKI                13
VEGF_IV_chemo                  9
PD-1/PD-L1_CTLA4_VEGF_IV       6
Name: count, dtype: int64

In [83]:
df_regimen_cat_pt

,mpi_id,regimen_biosimilar,AZ_category_use
6,112171181,"durvalumab,tremelimumab",PD-1/PD-L1_CTLA4
15,137244673,"durvalumab,tremelimumab",PD-1/PD-L1_CTLA4
25,244067723,"atezolizumab,bevacizumab",PD-1/PD-L1_VEGF_IV
26,252301890,"atezolizumab,bevacizumab",PD-1/PD-L1_VEGF_IV
27,255272953,sorafenib,TKI
...,...,...,...
14547,989569413,lenvatinib,TKI
14552,994201021,"cisplatin,durvalumab,gemcitabine,gemcitabine h...",PD-1/PD-L1_chemo
14553,995045745,atezolizumab,PD-1/PD-L1_mono
14561,997668559,nivolumab,PD-1/PD-L1_mono


In [84]:
count = df_demogr_use['last_cur_completion_date'].notnull().sum()
print("Count of non-null completion dates:", count)

Count of non-null completion dates: 889


In [ ]:
# Apply the same rare-category collapse to the full multi-LOT dataset (df_lot_use_cat)
# so the category labels are consistent across both the LOT1-only and all-LOT views.
df_lot_use_cat.loc[
    df_lot_use_cat['AZ_category_use'].isin(RARE_CATEGORIES), 'AZ_category_use'
] = 'other'

In [86]:
df_lot_use_cat['AZ_category_use'].value_counts()

AZ_category_use
PD-1/PD-L1_VEGF_IV              1309
PD-1/PD-L1_CTLA4                1050
TKI                              689
PD-1/PD-L1_mono                  344
other                            208
chemo                            170
PD-1/PD-L1_chemo                 106
PD-1/PD-L1_TKI                    34
PD-1/PD-L1_VEGF_IV_TKI            33
PD-1/PD-L1_other                  16
VEGF_IV_chemo                     13
PD-1/PD-L1_CTLA4_VEGF_IV           8
PD-1/PD-L1_CTLA4_VEGF_IV_TKI       1
Name: count, dtype: int64

In [87]:
df_disease_diag_use

,mpi_id,division_mask,combined_div_mpi_id,diag_date,cancer_code,cancer_stage,t_value,n_value,m_value,g_value,...,date_of_progression,metastatic_date,metastatic_site,disease_subtype,location,histology,crpc_diagnosis_date,rsi,condition_concept_id,mcrpc_date
0,355000716,316,316_355000716,2018-09-11,C22.0,IIIB,T4,N0,M0,NaN,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,D,35206146,NaN
1,356830665,316,316_356830665,2022-09-05,C22.0,IIIB,T4,N0,M0,NaN,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,D,35206146,NaN
2,357503542,316,316_357503542,2024-07-17,C22.0,II,T2,N0,M0,NaN,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,D,35206146,NaN
4,358644266,316,316_358644266,2024-03-01,C22.0,IIIA,T3,N0,M0,NaN,...,NaN,2024-03-01,NaN,NaN,Liver,Unknown,NaN,E,35206146,NaN
5,360849566,316,316_360849566,2020-01-01,C22.0,IIIA,T3,N0,M0,NaN,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,E,35206146,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21701,500387928,773,773_500387928,2022-10-18,C22.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Liver,NaN,NaN,NaN,35206146,NaN
21703,508166433,773,773_508166433,2016-12-24,C22.0,IV,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,D,35206146,NaN
21704,510665375,773,773_510665375,2025-05-01,C22.0,IVB,T,N,M1,NaN,...,NaN,2025-05-01,NaN,NaN,Liver,Unknown,NaN,D,35206146,NaN
21705,511952086,773,773_511952086,2022-10-18,C22.0,II,T2,NX,M0,NaN,...,NaN,NaN,NaN,NaN,Liver,Unknown,NaN,D,35206146,NaN


In [ ]:
# Attach HCC diagnosis date from df_disease to each LOT row.
# Used later to calculate time-to-treatment (diagnosis → LOT1 start).
# Left join preserves all LOT rows; patients without a disease record get NaT.
df_lot_use_cat = df_lot_use_cat.merge(
    df_disease_diag_use[['combined_div_mpi_id', 'diag_date']],
    on='combined_div_mpi_id',
    how='left'
)

In [89]:
df_lot_use_cat

,mpi_id,division_mask,combined_div_mpi_id,lot,regimen,start_date,end_date,duration_days,duration_months,metastatic,no_div_lot,regimen_biosimilar,AZ_category_use,diag_date
0,112171181,128,128_112171181,1,"durvalumab,tremelimumab-actl",2025-08-07,2025-10-30,84,2,1,1,"durvalumab,tremelimumab",PD-1/PD-L1_CTLA4,2024-12-16
1,137244673,128,128_137244673,1,"durvalumab,tremelimumab-actl",2023-12-07,2025-12-05,729,24,1,1,"durvalumab,tremelimumab",PD-1/PD-L1_CTLA4,2023-11-22
2,244067723,128,128_244067723,1,"atezolizumab,bevacizumab-bvzr",2023-03-08,2023-04-26,49,1,1,1,"atezolizumab,bevacizumab",PD-1/PD-L1_VEGF_IV,2022-06-02
3,252301890,128,128_252301890,1,"atezolizumab,bevacizumab-bvzr",2022-11-02,2023-06-21,231,7,1,1,"atezolizumab,bevacizumab",PD-1/PD-L1_VEGF_IV,2022-10-07
4,255272953,128,128_255272953,1,sorafenib,2023-07-11,2023-09-09,60,2,0,1,sorafenib,TKI,2023-06-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3976,995045745,997,997_995045745,1,atezolizumab,2023-04-03,2023-07-10,98,3,0,1,atezolizumab,PD-1/PD-L1_mono,2023-02-02
3977,995045745,997,997_995045745,2,"durvalumab,tremelimumab-actl",2023-08-02,2024-01-30,181,5,0,2,"durvalumab,tremelimumab",PD-1/PD-L1_CTLA4,2023-02-02
3978,995045745,997,997_995045745,3,lenvatinib,2024-02-20,2024-10-05,228,8,0,3,lenvatinib,TKI,2023-02-02
3979,997668559,997,997_997668559,1,nivolumab,2022-07-11,2022-10-04,85,3,0,1,nivolumab,PD-1/PD-L1_mono,2022-03-15


In [ ]:
df_lot_use_cat.to_excel(os.path.join(lot_output, 'dflotusecat.xlsx'), index=False)

In [ ]:
# Restrict visits to the 3,010-patient cohort to reduce memory footprint
df_visit = df_visit[df_visit['combined_div_mpi_id'].isin(df_lot_use['combined_div_mpi_id'])]

In [92]:
df_visit.head()

,mpi_id,division_mask,combined_div_mpi_id,visit_date,visit_type,visit_desc,rsi,visit_concept_id
0,634857065,532,532_634857065,2026-11-14,Visit Occurrence Record not tagged to other ca...,Uncategorized Visit,E,0
1,634857065,532,532_634857065,2026-11-21,Visit Occurrence Record not tagged to other ca...,Uncategorized Visit,E,0
2,634857065,532,532_634857065,2026-11-28,Visit Occurrence Record not tagged to other ca...,Uncategorized Visit,E,0
3,634857065,532,532_634857065,2026-12-05,Visit Occurrence Record not tagged to other ca...,Uncategorized Visit,E,0
4,634857065,532,532_634857065,2026-12-12,Visit Occurrence Record not tagged to other ca...,Uncategorized Visit,E,0


In [93]:
df_lot_use_cat = df_lot_use_cat

In [ ]:
# Restrict to LOT1 rows only for the primary analysis table (3,010 patients, one row each)
df_lot_use_cat = df_lot_use_cat[df_lot_use_cat['no_div_lot'] == 1]

In [95]:
df_lot_use_cat

,mpi_id,division_mask,combined_div_mpi_id,lot,regimen,start_date,end_date,duration_days,duration_months,metastatic,no_div_lot,regimen_biosimilar,AZ_category_use,diag_date
0,112171181,128,128_112171181,1,"durvalumab,tremelimumab-actl",2025-08-07,2025-10-30,84,2,1,1,"durvalumab,tremelimumab",PD-1/PD-L1_CTLA4,2024-12-16
1,137244673,128,128_137244673,1,"durvalumab,tremelimumab-actl",2023-12-07,2025-12-05,729,24,1,1,"durvalumab,tremelimumab",PD-1/PD-L1_CTLA4,2023-11-22
2,244067723,128,128_244067723,1,"atezolizumab,bevacizumab-bvzr",2023-03-08,2023-04-26,49,1,1,1,"atezolizumab,bevacizumab",PD-1/PD-L1_VEGF_IV,2022-06-02
3,252301890,128,128_252301890,1,"atezolizumab,bevacizumab-bvzr",2022-11-02,2023-06-21,231,7,1,1,"atezolizumab,bevacizumab",PD-1/PD-L1_VEGF_IV,2022-10-07
4,255272953,128,128_255272953,1,sorafenib,2023-07-11,2023-09-09,60,2,0,1,sorafenib,TKI,2023-06-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3974,989569413,997,997_989569413,1,lenvatinib,2024-07-19,2025-01-03,168,6,0,1,lenvatinib,TKI,2024-07-08
3975,994201021,997,997_994201021,1,"cisplatin,durvalumab,gemcitabine,gemcitabine h...",2023-10-04,2025-09-02,699,23,0,1,"cisplatin,durvalumab,gemcitabine,gemcitabine h...",PD-1/PD-L1_chemo,2023-08-23
3976,995045745,997,997_995045745,1,atezolizumab,2023-04-03,2023-07-10,98,3,0,1,atezolizumab,PD-1/PD-L1_mono,2023-02-02
3979,997668559,997,997_997668559,1,nivolumab,2022-07-11,2022-10-04,85,3,0,1,nivolumab,PD-1/PD-L1_mono,2022-03-15


In [ ]:
df_lot_use_cat.to_excel(os.path.join(lot_output, 'df_lot_use_cat.xlsx'), index=False)

## Curation Patient List

In [ ]:
import datetime

# ── Build clean curation patient list ─────────────────────────────────────────
# Select and rename columns to human-readable names for the curation team.
# This is the deliverable list — one row per patient, 3,010 patients total.
df_curation_list = df_lot_use_cat[[
    'mpi_id',
    'division_mask',
    'combined_div_mpi_id',
    'diag_date',
    'regimen_biosimilar',
    'AZ_category_use',
    'start_date',
    'end_date',
    'duration_months',
    'metastatic',
]].copy().reset_index(drop=True)

df_curation_list.rename(columns={
    'diag_date':         'hcc_diagnosis_date',
    'start_date':        'lot1_start_date',
    'end_date':          'lot1_end_date',
    'duration_months':   'lot1_duration_months',
    'AZ_category_use':   'treatment_category',
    'regimen_biosimilar':'regimen',
}, inplace=True)

print(f"Total patients for curation: {df_curation_list['mpi_id'].nunique()}")
df_curation_list.head()

In [ ]:
# Save curation list — filename includes today's date for version tracking
today = datetime.date.today().strftime('%Y%m%d')
curation_filename = f'HCC_curation_patient_list_{today}.xlsx'
df_curation_list.to_excel(os.path.join(lot_output, curation_filename), index=False)
print(f"Saved: {curation_filename} ({len(df_curation_list)} patients)")